# fhr-monitor-analyzer demo

This notebook shows the basic Python library workflow:

1. Create or load fetal monitor CSV data.
2. Analyze the data with `fhr_monitor_analyzer`.
3. Inspect the JSON report.
4. Render a fetal HR / maternal HR / TOCO diagram.

For local development, run this first from the repository root:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip maturin matplotlib notebook ipykernel
maturin build --features python --out dist
python -m pip install --force-reinstall dist/*.whl
python -m ipykernel install --user --name fhr-monitor-analyzer --display-name "Python (fhr-monitor-analyzer)"
python -m jupyter notebook examples/fhr_monitor_analyzer_demo.ipynb
```

The analyzer is decision support only. The output should be reviewed as structured tracing metadata, not as an autonomous diagnosis or treatment instruction.

If the import cell fails, check that the notebook is using the `Python (fhr-monitor-analyzer)` kernel.

In [ ]:
from __future__ import annotations

import csv
import json
import math
from datetime import datetime, timedelta
from pathlib import Path

from IPython.display import Image, display

import fhr_monitor_analyzer as analyzer

## Create demo monitor data

The production caller would usually provide real monitor data. For a self-contained notebook, this cell creates a synthetic 24-minute CSV with `Date`, `HR1`, `HRM`, and `TOCO` columns. The shape intentionally includes baseline variation, two accelerations, several contractions, and two variable-like decelerations so the report has meaningful features to inspect.

In [ ]:
examples_dir = Path.cwd()
csv_path = examples_dir / "demo_monitor.csv"
plot_path = examples_dir / "demo_monitor_plot.png"

start = datetime(2026, 5, 12, 11, 52, 35, 52000)
rows = []

for second in range(24 * 60):
    t = start + timedelta(seconds=second)
    fetal_hr = 135 + 6 * math.sin(second / 9.0) + 3 * math.sin(second / 31.0)

    if 8 * 60 <= second <= 8 * 60 + 22:
        fetal_hr += 22 * math.sin(math.pi * (second - 8 * 60) / 22)
    if 17 * 60 + 30 <= second <= 17 * 60 + 52:
        fetal_hr += 20 * math.sin(math.pi * (second - (17 * 60 + 30)) / 22)
    if 11 * 60 + 25 <= second <= 11 * 60 + 55:
        fetal_hr -= 42 * math.sin(math.pi * (second - (11 * 60 + 25)) / 30)
    if 18 * 60 + 10 <= second <= 18 * 60 + 42:
        fetal_hr -= 48 * math.sin(math.pi * (second - (18 * 60 + 10)) / 32)

    hrm = 95 + 4 * math.sin(second / 47.0)
    toco = 9 + 2 * math.sin(second / 18.0)
    for peak in [3 * 60, 7 * 60, 11 * 60 + 45, 15 * 60 + 30, 19 * 60]:
        distance = abs(second - peak)
        if distance <= 38:
            toco += 42 * (1 + math.cos(math.pi * distance / 38)) / 2

    rows.append({
        "Date": t.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3],
        "HR1": round(fetal_hr, 2),
        "HRM": round(hrm, 2),
        "TOCO": round(toco, 2),
    })

with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Date", "HR1", "HRM", "TOCO"])
    writer.writeheader()
    writer.writerows(rows)

csv_path, len(rows)

## Analyze CSV data

The analysis API returns a JSON string. This mirrors the CLI and the future service response, so applications can pass it through directly or parse it with `json.loads`.

In [ ]:
report_json = analyzer.analyze_csv_file(csv_path, channel="HR1")
report = json.loads(report_json)

print(json.dumps(report, indent=2)[:4000])

## Inspect the current window

The report contains one or more windows. Chunk mode returns the current-state window. Rolling replay mode can return multiple windows when `window_min` and `step_sec` are supplied.

In [ ]:
window = report["windows"][-1]
features = window["features"]

summary = {
    "classification": window["classification"],
    "alert_level": window["alert_level"],
    "baseline_bpm": window["baseline_bpm"],
    "baseline_class": window["baseline_class"],
    "variability_bpm": window["variability_bpm"],
    "variability_class": window["variability_class"],
    "acceleration_count": features["acceleration_count"],
    "deceleration_count": features["deceleration_count"],
    "total_deceleration_seconds": features["total_deceleration_seconds"],
    "contractions_per_10_min": features["contractions_per_10_min"],
    "fetal_usable_ratio": window["data_quality"]["fetal_usable_ratio"],
    "reasons": window["reasons"],
    "high_risk_features": window["high_risk_features"],
    "protective_features": window["protective_features"],
    "limitations": window["limitations"],
}

summary

## Render the monitor diagram

`plot_csv_file` writes a PNG with fetal HR, maternal HR, and TOCO panels. The same logic is used by `scripts/plot_monitor_csv.py`.

In [ ]:
written_plot = analyzer.plot_csv_file(
    csv_path,
    output=plot_path,
    channel="HR1",
    title="fhr-monitor-analyzer demo tracing",
)

display(Image(filename=written_plot))

## Analyze a JSON request

The service-facing request format uses `samples[*].t`, `hr1`, `hrm`, and `toco`. This example converts the same generated rows into a JSON request and analyzes it through `analyze_json`.

In [ ]:
request = {
    "episode_id": "demo-episode",
    "sent_at": datetime.utcnow().isoformat(timespec="milliseconds") + "Z",
    "analysis_options": {"fetal_channel": "HR1", "max_analysis_minutes": 30},
    "metadata": {"gestational_age_weeks": 39},
    "samples": [
        {
            "t": row["Date"].replace(" ", "T") + "Z",
            "hr1": row["HR1"],
            "hrm": row["HRM"],
            "toco": row["TOCO"],
        }
        for row in rows
    ],
}

json_report = json.loads(analyzer.analyze_json(json.dumps(request)))
json_report["windows"][-1]["alert_level"], json_report["windows"][-1]["classification"]